# 02 — Preprocess & align to the 1 km target grid

Produce the **prioritizr-ready feature stack** on the shared analysis grid
(ESRI:102008, 1 km, Y2Y boundary buffered by `BUFFER_KM`). Outputs are oriented,
non-normalized, masked COGs written to `input_data/aligned_stack/` (the R hand-off).

**This notebook reads and writes full pixel data** (unlike `01`, which is metadata-only)
and can run for a while. Ethan runs it; Claude never executes cells.

**Two stages:**

- **Stage 1 — warp** (system **`gdalwarp`**): one streamed reproject + resample +
  cutline-clip per layer to `cleaned_aligned/` (intermediate, *raw orientation*). Reads
  only the Y2Y window so global rasters are never warped in full. Resampling per layer
  from `config.py`: `average` (down-sample density), `bilinear` (up-sample coarse),
  `nearest` (categorical EFGs).
- **Stage 2 — orient → mask → QA → COG** (numpy + rasterio, in memory; the 1 km grid is
  small): apply value orientation so **higher = more conservation value** (gHM→intactness
  `1−gHM`; backward velocity→refugia `vmax−v`); rasterize the PA mask; build **one
  planning-unit (PU) mask** from the continuous features and apply it identically to every
  feature **and** the uniform cost layer; QA the carbon tail and connectivity distribution
  (surface, don't silently transform); write COGs; validate the whole stack.

**Decisions encoded here** follow the pre-processing hand-off (orientation, no
normalization, raw carbon, PU-mask consistency, NoData→NA) with **resolution held at 1 km**
for iteration 1 per `CLAUDE.md`. Continuous-feature values are kept **native** (oriented,
non-negative) — the target-based model makes cross-feature normalization unnecessary.

**Kernel:** select **`Python (y2y-geo)`** (the project `.venv`, Python 3.12).

In [1]:
import importlib
import math
import subprocess
import shutil
import time

import numpy as np
import pyproj
import rasterio
import geopandas as gpd

import config
importlib.reload(config)  # pick up edits to config.py when re-running this cell
from config import (
    DATASETS, INPUT_DIR, ALIGNED_DIR, HANDOFF_DIR, CORRIDOR_REF, PA_VECTOR,
    TARGET_CRS, TARGET_RES_M, BUFFER_KM,
    CONNECTIVITY_CAP_PCTILE, CARBON_FLAG_PCTILE,
    find_rasters, pick_representative, study_area,
)

# This notebook drives the system GDAL CLIs (osgeo bindings aren't in the venv).
for cli in ("gdalbuildvrt", "gdalwarp", "gdal_rasterize"):
    assert shutil.which(cli), f"{cli} not found on PATH (need system GDAL CLIs)"
print("GDAL CLIs OK | rasterio", rasterio.__version__, "| geopandas", gpd.__version__)
print("datasets in config:", len(DATASETS))

GDAL CLIs OK | rasterio 1.5.0 | geopandas 1.1.3
datasets in config: 10


In [3]:
# ---- Study area + 1 km target grid ---------------------------------------
ALIGNED_DIR.mkdir(parents=True, exist_ok=True)   # stage-1 intermediates
HANDOFF_DIR.mkdir(parents=True, exist_ok=True)   # stage-2 final COGs (R reads these)

sa = study_area(BUFFER_KM)  # Y2Y boundary, ESRI:102008, buffered by BUFFER_KM
res = TARGET_RES_M
minx, miny, maxx, maxy = sa.total_bounds
left = math.floor(minx / res) * res  # snap extent to a clean 1 km grid
bottom = math.floor(miny / res) * res
right = math.ceil(maxx / res) * res
top = math.ceil(maxy / res) * res
TE = [left, bottom, right, top]  # target extent, in TARGET_CRS (gdalwarp -te)
WIDTH = int(round((right - left) / res))
HEIGHT = int(round((top - bottom) / res))
TRANSFORM = rasterio.transform.from_origin(left, top, res, res)

# Reference extent for any aggregation / cap / orientation (documented per hand-off).
print(f"REFERENCE EXTENT (TARGET_CRS {TARGET_CRS}): {TE}  -> {WIDTH} x {HEIGHT} cells @ {res} m")

# Cutline used to clip every layer to the buffered corridor.
CUTLINE_PATH = ALIGNED_DIR / "_study_area.gpkg"
sa.to_file(CUTLINE_PATH, driver="GPKG")
print(f"cutline     -> {CUTLINE_PATH.relative_to(INPUT_DIR.parent)}")

REFERENCE EXTENT (TARGET_CRS ESRI:102008): [-2206000, 273000, -920000, 3585000]  -> 1286 x 3312 cells @ 1000 m
cutline     -> input_data/cleaned_aligned/_study_area.gpkg


## Stage 1 — warp to grid (intermediate, raw orientation)

Reproject + resample + cutline-clip every source to `cleaned_aligned/`. These are
*intermediate* products with **raw** values (no orientation/mask yet); Stage 2 turns
them into the final hand-off stack.

In [4]:
# ---- Rebuild the gHM mosaic VRT from its tiles (recorded in-workflow) -----
# The original VRT was deleted; we rebuild it here so the mosaic is reproducible.
hm = DATASETS["human_modification"]
hm_tiles = find_rasters(hm)  # the 4 gHM GeoTIFF tiles
HM_VRT = hm["path"] / "HM_Y2Y_2024_90_60land.vrt"
proc = subprocess.run(
    ["gdalbuildvrt", str(HM_VRT), *map(str, hm_tiles)],
    capture_output=True, text=True,
)
if proc.returncode != 0:
    raise RuntimeError(f"gdalbuildvrt failed:\n{proc.stderr}")
print(f"built VRT from {len(hm_tiles)} tiles -> {HM_VRT.name}")

built VRT from 4 tiles -> HM_Y2Y_2024_90_60land.vrt


In [5]:
# ---- Alignment helpers ---------------------------------------------------
GDAL_RESAMP = {"average": "average", "bilinear": "bilinear", "nearest": "near"}


def pick_nodata(src):
    """Output NoData: keep the source's; else NaN (float) / 0 (integer)."""
    if src.nodata is not None:
        return src.nodata
    return float("nan") if np.issubdtype(np.dtype(src.dtypes[0]), np.floating) else 0


def source_for(name, cfg):
    """The raster to align: the rebuilt VRT for gHM, else the representative."""
    if cfg.get("build_vrt"):
        return HM_VRT
    return pick_representative(cfg, find_rasters(cfg))


def warp_to_grid(src, dst, resampling, nodata):
    """Reproject + resample + clip a raster to the shared 1 km grid via gdalwarp.
    gdalwarp reads only the source window covering the target extent, so global
    rasters are never warped in full (clip-before-reproject is implicit). The
    cutline masks everything outside the buffered corridor to NoData while the
    fixed -te/-tr keeps every output on the identical grid."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "gdalwarp", "-overwrite",
        "-t_srs", TARGET_CRS,
        "-te", *map(str, TE),
        "-tr", str(TARGET_RES_M), str(TARGET_RES_M),
        "-r", GDAL_RESAMP[resampling],
        "-cutline", str(CUTLINE_PATH),
        "-dstnodata", str(nodata),
        "-of", "GTiff",
        "-co", "TILED=YES", "-co", "COMPRESS=DEFLATE", "-co", "BIGTIFF=IF_SAFER",
        "-multi", "-wo", "NUM_THREADS=ALL_CPUS",
        str(src), str(dst),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"gdalwarp failed for {src}:\n{proc.stderr}")


def align_dataset(name, cfg, dst):
    """Warp one dataset's source raster onto the shared grid."""
    src = source_for(name, cfg)
    with rasterio.open(src) as s:
        nd = pick_nodata(s)
    warp_to_grid(src, dst, cfg["resampling"], nd)
    return dst

In [6]:
# ---- Align all single-file datasets --------------------------------------
single = {n: c for n, c in DATASETS.items() if not c.get("multi")}
for name, cfg in single.items():
    t0 = time.perf_counter()
    dst = ALIGNED_DIR / f"{name}.tif"
    align_dataset(name, cfg, dst)
    print(f"[{name:30s}] {cfg['resampling']:8s} -> {dst.name}  ({time.perf_counter() - t0:5.1f}s)")
print("single-file datasets aligned.")

[human_modification            ] average  -> human_modification.tif  ( 60.5s)
[transboundary_connectivity    ] average  -> transboundary_connectivity.tif  (  2.9s)
[climate_corridors             ] bilinear -> climate_corridors.tif  (  0.6s)
[climate_type_macrorefugia     ] bilinear -> climate_type_macrorefugia.tif  (  0.8s)
[irrecoverable_carbon_biomass  ] average  -> irrecoverable_carbon_biomass.tif  (  0.8s)
[irrecoverable_carbon_m_soc    ] average  -> irrecoverable_carbon_m_soc.tif  (  0.8s)
[irrecoverable_carbon_sl_soc   ] average  -> irrecoverable_carbon_sl_soc.tif  (  0.7s)
[aoh_richness_mammals          ] average  -> aoh_richness_mammals.tif  (243.4s)
[aoh_richness_birds            ] average  -> aoh_richness_birds.tif  (269.5s)
single-file datasets aligned.


In [7]:
# ---- IUCN EFGs: align every grid that actually occurs in the study area ---
# EFG maps are continental/global in extent, so a bbox test always "overlaps".
# Real overlap = the EFG has present cells (value > 0) inside the corridor after
# clipping. We warp each (cheap -- they're coarse), then drop any with no presence.
# NOTE: assumes EFG values encode occurrence as > 0 (0/nodata = absent); verify the
# value scheme of the GET maps if the kept/dropped counts look off.
efg = DATASETS["iucn_efg"]
efg_rasters = find_rasters(efg)
efg_out = ALIGNED_DIR / "iucn_efg"
kept, dropped = [], []
for i, src in enumerate(efg_rasters, 1):
    dst = efg_out / f"{src.stem}.tif"
    with rasterio.open(src) as s:
        nd = pick_nodata(s)
    warp_to_grid(src, dst, efg["resampling"], nd)
    with rasterio.open(dst) as s:
        arr = s.read(1, masked=True)
    m = np.ma.getmaskarray(arr)
    present = int(((arr.data > 0) & ~m).sum())
    if present == 0:
        dst.unlink()  # no occurrence in Y2Y -> not a feature for this study area
        dropped.append(src.stem)
    else:
        kept.append((src.stem, present))
    if i % 20 == 0:
        print(f"  ...processed {i}/{len(efg_rasters)} EFGs")

print(f"EFGs: kept {len(kept)} / {len(efg_rasters)} (dropped {len(dropped)} with no Y2Y presence)")
print("dropped:", ", ".join(dropped) if dropped else "(none)")

  ...processed 20/109 EFGs
  ...processed 40/109 EFGs
  ...processed 60/109 EFGs
  ...processed 80/109 EFGs
  ...processed 100/109 EFGs
EFGs: kept 42 / 109 (dropped 67 with no Y2Y presence)
dropped: F1.5.web.map_v1.0, F1.7.web.orig_v2.0, F2.5.web.mix_v1.0, F2.8.web.mix_v1.0, F3.3.web.alt_v2.0, FM1.1.web.map_v1.0, FM1.2.web.map_v1.0, FM1.3.web.orig_v2.0, M1.1.web.orig_v1.0, M1.10.WM.nwx_v1.0, M1.2.web.orig_v2.2, M1.3.web.orig_v1.0, M1.4.web.orig_v1.0, M1.5.web.orig_v1.0, M1.6.web.orig_v1.0, M1.7.web.orig_v1.0, M1.8.web.orig_v1.0, M1.9.web.orig_v1.0, M2.1.web.orig_v2.1, M2.2.web.orig_v2.0, M2.3.web.orig_v1.0, M2.4.web.orig_v1.0, M2.5.web.orig_v1.0, M3.1.web.orig_v1.0, M3.2.web.orig_v1.0, M3.3.web.orig_v1.0, M3.4.web.orig_v2.0, M3.5.web.orig_v1.0, M3.6.web.orig_v1.0, M3.7.web.orig_v1.0, M4.1.web.orig_v1.0, M4.2.web.orig_v1.0, MFT1.1.map.orig_v1.0, MFT1.2.web.orig_v2.0, MFT1.3.web.orig_v2.0, MT1.1.web.map_v1.0, MT1.2.IM.orig_v1.0, MT1.3.web.map_v1.0, MT1.4.web.map_v1.0, MT2.1.web.map_v1.0,

## Stage 2 — orient → mask → QA → COG hand-off

The 1 km grid is small (~4 M cells), so Stage 2 works in memory with numpy + rasterio.
Steps: load the warped intermediates → apply orientation (higher = more conservation
value) → rasterize the PA mask → build one planning-unit (PU) mask from the continuous
features → QA carbon/connectivity tails → apply the PU mask to every feature **and** the
uniform cost layer → write COGs to `aligned_stack/`.

In [8]:
# ---- Stage-2 helpers -----------------------------------------------------
from rasterio.io import MemoryFile
import rasterio.shutil as rio_shutil

# Continuous features are the single-file datasets (each one prioritizr feature).
CONT_FEATURES = [n for n, c in DATASETS.items() if not c.get("multi")]


def read_grid(path, fill=np.nan):
    """Read a stage-1 intermediate as float32 with NoData -> `fill` (NaN by default)."""
    with rasterio.open(path) as s:
        a = s.read(1, masked=True).astype("float32")
    return a.filled(fill)


def write_cog(path, arr, nodata, dtype):
    """Write a single-band *true* COG on the canonical grid.

    GDAL's COG driver is CreateCopy-only, so we build the array in an in-memory
    GTiff then copy it to a COG (this is what actually produces the COG layout +
    overviews; a direct driver='COG' open silently yields a plain GTiff)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    ov = "nearest" if np.dtype(dtype) == np.uint8 else "average"  # categorical vs continuous
    prof = dict(driver="GTiff", width=WIDTH, height=HEIGHT, count=1, dtype=dtype,
                crs=TARGET_CRS, transform=TRANSFORM, nodata=nodata)
    with MemoryFile() as mem:
        with mem.open(**prof) as tmp:
            tmp.write(np.asarray(arr).astype(dtype), 1)
            rio_shutil.copy(tmp, path, driver="COG", compress="DEFLATE",
                            overview_resampling=ov, BIGTIFF="IF_SAFER")


EFG_NODATA = 255   # uint8 sentinel: keeps 0 (absent) a VALID feature value
PA_NODATA = 255
print(f"continuous features ({len(CONT_FEATURES)}):", ", ".join(CONT_FEATURES))

continuous features (9): human_modification, transboundary_connectivity, climate_corridors, climate_type_macrorefugia, irrecoverable_carbon_biomass, irrecoverable_carbon_m_soc, irrecoverable_carbon_sl_soc, aoh_richness_mammals, aoh_richness_birds


In [9]:
# ---- Orient every feature so HIGHER = more conservation value ------------
# complement: 1 - gHM (intactness, x in [0,1])   invert: vmax - v (refugial value)
# others: already "more = better", kept raw. All must be non-negative afterwards.
feat = {}        # name -> oriented float32 array (NaN = NoData)
orient_log = []
for name in CONT_FEATURES:
    a = read_grid(ALIGNED_DIR / f"{name}.tif")
    o = DATASETS[name].get("orient")
    if o == "complement":
        a = np.clip(1.0 - a, 0.0, 1.0)          # tiny gHM under/overshoot clipped to [0,1]
        note = "1 - gHM -> intactness, clipped [0,1]"
    elif o == "invert":
        vmax = float(np.nanmax(a))
        a = vmax - a                            # low velocity -> high refugial value
        note = f"vmax - v -> refugia, vmax={vmax:.4f} over reference extent"
    else:
        note = "raw (already more = better)"
    feat[name] = a.astype("float32")
    vmin = float(np.nanmin(a))
    orient_log.append((name, o or "none", vmin, float(np.nanmax(a)), note))

print(f"{'feature':32s} {'orient':10s} {'min':>10s} {'max':>12s}  note")
for name, o, lo, hi, note in orient_log:
    print(f"{name:32s} {o:10s} {lo:10.4f} {hi:12.4f}  {note}")

# Non-negativity guard (allow a hair of float noise, then floor at 0).
neg = {n: int(np.nansum(a < -1e-6)) for n, a in feat.items()}
assert not any(neg.values()), f"negative values after orientation: {neg}"
for n in feat:
    feat[n] = np.clip(feat[n], 0.0, None)
print("orientation OK -> all features non-negative.")

/Users/ethanberman/Dropbox/Python Projects/y2y-spatial-optimization/.venv/lib/python3.12/site-packages/numpy/ma/core.py:498: RuntimeWarning: overflow encountered in cast
  fill_value = np.asarray(fill_value, dtype=ndtype)


feature                          orient            min          max  note
human_modification               complement     0.0207       1.0000  1 - gHM -> intactness, clipped [0,1]
transboundary_connectivity       none           0.0002      74.9116  raw (already more = better)
climate_corridors                none           0.0546      21.2592  raw (already more = better)
climate_type_macrorefugia        invert         0.0000      16.3806  vmax - v -> refugia, vmax=16.4774 over reference extent
irrecoverable_carbon_biomass     none           0.0000     180.8005  raw (already more = better)
irrecoverable_carbon_m_soc       none           0.0000    2255.0642  raw (already more = better)
irrecoverable_carbon_sl_soc      none           0.0000     404.7220  raw (already more = better)
aoh_richness_mammals             none          12.0000      54.0000  raw (already more = better)
aoh_richness_birds               none          13.0000     115.0000  raw (already more = better)
orientation OK -

In [10]:
# ---- Rasterize the protected-areas mask onto the grid --------------------
# PA polygons are already ESRI:102008 (no reproject); burn 1 = protected, 0 = not.
# This is prep only -- its use as a lock-in constraint is R-side.
PA_RASTER = ALIGNED_DIR / "_pa_mask.tif"
cmd = [
    "gdal_rasterize", "-burn", "1", "-init", "0", "-ot", "Byte",
    "-te", *map(str, TE), "-tr", str(res), str(res),
    "-of", "GTiff", "-co", "TILED=YES", "-co", "COMPRESS=DEFLATE",
    str(PA_VECTOR), str(PA_RASTER),
]
proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    raise RuntimeError(f"gdal_rasterize failed:\n{proc.stderr}")
pa = read_grid(PA_RASTER, fill=0).astype("uint8")   # 0/1 on the grid
print(f"PA mask: {int((pa == 1).sum()):,} protected cells on the grid")

PA mask: 201,768 protected cells on the grid


In [11]:
# ---- One planning-unit (PU) mask, applied identically to every layer ------
# PU = cells valid in ALL continuous features (and inside the corridor, since
# outside-corridor cells were warped to NoData -> non-finite). EFG absence is the
# value 0, NOT NoData, so EFGs do NOT constrain the PU. The cost layer and every
# feature get this same mask -> no cell valid in one layer but NoData in another.
valid = {n: np.isfinite(a) for n, a in feat.items()}
PU = np.logical_and.reduce(list(valid.values()))
n_pu = int(PU.sum())

print(f"{'feature':32s} {'valid cells':>14s}  {'% of grid':>10s}")
for n, v in valid.items():
    c = int(v.sum())
    print(f"{n:32s} {c:14,}  {100 * c / (WIDTH * HEIGHT):9.2f}%")
print("-" * 60)
print(f"{'PLANNING UNIT (intersection)':32s} {n_pu:14,}  {100 * n_pu / (WIDTH * HEIGHT):9.2f}%")

# Surface the layer(s) that most constrain the PU (valid but dropped by the AND).
limiting = sorted(((int((v & ~PU).sum()), n) for n, v in valid.items()), reverse=True)
print("most-limiting layers (valid cells lost to the intersection):")
for lost, n in limiting[:3]:
    print(f"  {n:32s} drops {lost:,} cells")

feature                             valid cells   % of grid
human_modification                    1,557,264      36.56%
transboundary_connectivity            1,556,916      36.55%
climate_corridors                     1,551,614      36.43%
climate_type_macrorefugia             1,551,621      36.43%
irrecoverable_carbon_biomass          1,274,564      29.92%
irrecoverable_carbon_m_soc            1,355,145      31.82%
irrecoverable_carbon_sl_soc           1,355,145      31.82%
aoh_richness_mammals                  1,555,153      36.51%
aoh_richness_birds                    1,555,153      36.51%
------------------------------------------------------------
PLANNING UNIT (intersection)          1,272,914      29.89%
most-limiting layers (valid cells lost to the intersection):
  human_modification               drops 284,350 cells
  transboundary_connectivity       drops 284,002 cells
  aoh_richness_mammals             drops 282,239 cells


In [12]:
# ---- QA: surface the carbon & connectivity tails (don't silently transform) --
def quantiles(a, qs=(50, 90, 99, 99.9, 100)):
    return {q: float(np.nanpercentile(a, q)) for q in qs}


# Carbon: FLAG implausibly high cells for review; do NOT transform the surface.
print("CARBON tail QA (flag-only; winsorize confirmed artefacts later):")
for name in [n for n in CONT_FEATURES if "carbon" in n]:
    vals = feat[name][PU]
    thr = float(np.nanpercentile(vals, CARBON_FLAG_PCTILE * 100))
    n_flag = int(np.nansum((feat[name] > thr) & PU))
    q = quantiles(vals)
    print(f"  {name:30s} p50={q[50]:.2f} p99={q[99]:.2f} p99.9={q[99.9]:.2f} "
          f"max={q[100]:.2f} | >{CARBON_FLAG_PCTILE:.3%}={thr:.2f}: {n_flag:,} cells flagged")

# Connectivity: expose the distribution; cap ONLY if a percentile is set in config.
conn = "transboundary_connectivity"
cvals = feat[conn][PU]
q = quantiles(cvals)
print(f"\nCONNECTIVITY distribution (current-flow, oriented raw): "
      f"p50={q[50]:.2f} p90={q[90]:.2f} p99={q[99]:.2f} p99.9={q[99.9]:.2f} max={q[100]:.2f}")
if CONNECTIVITY_CAP_PCTILE is not None:
    cap = float(np.nanpercentile(cvals, CONNECTIVITY_CAP_PCTILE * 100))
    n_capped = int(np.nansum((feat[conn] > cap) & PU))
    feat[conn] = np.clip(feat[conn], None, cap)
    print(f"  CAP APPLIED at p{CONNECTIVITY_CAP_PCTILE:.3%} = {cap:.2f} "
          f"({n_capped:,} pinch-point cells capped)")
else:
    print("  no cap applied (CONNECTIVITY_CAP_PCTILE=None) -- inspect the tail, "
          "then set the percentile in config.py if a cap is warranted.")

CARBON tail QA (flag-only; winsorize confirmed artefacts later):
  irrecoverable_carbon_biomass   p50=8.21 p99=108.17 p99.9=136.84 max=180.80 | >99.900%=136.84: 1,273 cells flagged
  irrecoverable_carbon_m_soc     p50=11.98 p99=571.02 p99.9=1123.95 max=2255.06 | >99.900%=1123.95: 1,273 cells flagged
  irrecoverable_carbon_sl_soc    p50=1.47 p99=71.94 p99.9=126.52 max=366.42 | >99.900%=126.52: 1,273 cells flagged

CONNECTIVITY distribution (current-flow, oriented raw): p50=1.65 p90=3.49 p99=6.20 p99.9=11.13 max=74.91
  no cap applied (CONNECTIVITY_CAP_PCTILE=None) -- inspect the tail, then set the percentile in config.py if a cap is warranted.


In [13]:
# ---- Apply the PU mask and write the final COG stack ---------------------
# Continuous features + cost: float32, NaN outside PU (reads as NA in R).
# EFGs + PA: uint8, 0 stays a valid value, 255 = outside PU.
efg_handoff = HANDOFF_DIR / "iucn_efg"
if efg_handoff.exists():
    for old in efg_handoff.glob("*.tif"):
        old.unlink()   # clear stale EFGs so a re-run reflects the current PU-based drops
n_written = 0

# Continuous features (oriented, masked, non-normalized, raw magnitudes kept).
for name, a in feat.items():
    out = np.where(PU, a, np.nan).astype("float32")
    write_cog(HANDOFF_DIR / f"{name}.tif", out, np.nan, "float32")
    n_written += 1

# Uniform area-based cost = 1 per planning unit (no real cost surface yet).
cost = np.where(PU, 1.0, np.nan).astype("float32")
write_cog(HANDOFF_DIR / "cost_uniform.tif", cost, np.nan, "float32")
n_written += 1

# Protected-areas lock-in mask, on the same PU.
pa_out = np.where(PU, pa, PA_NODATA).astype("uint8")
write_cog(HANDOFF_DIR / "mask_protected_areas.tif", pa_out, PA_NODATA, "uint8")
n_written += 1

# EFG features: 0 = absent (valid), 1 = minor, 2 = major, 255 = outside PU.
# An EFG with only 0s inside the PU has no occurrence in the study area -> its range
# doesn't overlap; DROP it (0 is a valid value, but a layer with no 1/2 is not a feature).
efg_intermediates = sorted((ALIGNED_DIR / "iucn_efg").glob("*.tif"))
n_efg, dropped_no_pu = 0, []
for src in efg_intermediates:
    raw = read_grid(src, fill=0)                      # source NoData(0) -> absent 0
    out = np.where(PU, np.rint(raw), EFG_NODATA).astype("uint8")
    present = int(((out == 1) | (out == 2)).sum())   # occurrence inside the PU
    if present == 0:
        dropped_no_pu.append(src.stem)
        continue                                     # range doesn't reach the study area
    write_cog(efg_handoff / src.name, out, EFG_NODATA, "uint8")
    n_efg += 1
    n_written += 1

print(f"wrote {n_written} COGs -> {HANDOFF_DIR.relative_to(INPUT_DIR.parent)}")
print(f"  {len(feat)} continuous features + cost + PA mask + {n_efg} EFGs")
if dropped_no_pu:
    print(f"dropped {len(dropped_no_pu)} EFG(s) with no occurrence (1/2) inside the PU: "
          f"{', '.join(dropped_no_pu)}")

wrote 51 COGs -> input_data/aligned_stack
  9 continuous features + cost + PA mask + 40 EFGs
dropped 2 EFG(s) with no occurrence (1/2) inside the PU: F2.7.web.mix_v1.0, MT2.2.IM.orig_v1.0


In [14]:
# ---- Validate the hand-off stack -----------------------------------------
# Identical grid, consistent NoData, matching PU cell counts, non-negativity,
# and an orientation spot-check. Fail loudly on any violation.
def _same(t1, t2):
    return all(abs(a - b) < 1e-6 for a, b in zip(t1, t2))


target_crs_obj = pyproj.CRS.from_user_input(TARGET_CRS)
ref_t = tuple(TRANSFORM)[:6]
cont_paths = [HANDOFF_DIR / f"{n}.tif" for n in CONT_FEATURES] + [HANDOFF_DIR / "cost_uniform.tif"]
u8_paths = [HANDOFF_DIR / "mask_protected_areas.tif"] + sorted((HANDOFF_DIR / "iucn_efg").glob("*.tif"))
problems = []

for p in cont_paths + u8_paths:
    with rasterio.open(p) as s:
        a = s.read(1, masked=True)
        n_valid = int((~np.ma.getmaskarray(a)).sum())
        is_u8 = p in u8_paths
        # grid identity
        if not (s.width == WIDTH and s.height == HEIGHT
                and _same(tuple(s.transform)[:6], ref_t)
                and s.crs is not None and pyproj.CRS.from_user_input(s.crs) == target_crs_obj):
            problems.append(f"{p.name}: grid mismatch")
        # NoData consistency
        nd_ok = (s.nodata == PA_NODATA) if is_u8 else (s.nodata is not None and np.isnan(s.nodata))
        if not nd_ok:
            problems.append(f"{p.name}: unexpected NoData {s.nodata}")
        # cell-count consistency: every layer covers exactly the PU
        if n_valid != n_pu:
            problems.append(f"{p.name}: {n_valid:,} valid != PU {n_pu:,}")
        # non-negativity (continuous features + EFG codes are all >= 0)
        if float(a.min()) < 0:
            problems.append(f"{p.name}: negative values present")

# Orientation spot-check: bounded intactness, non-negative refugia, and protected
# land should read as more intact on average (sanity, not a hard assert).
with rasterio.open(HANDOFF_DIR / "human_modification.tif") as s:
    intact = s.read(1, masked=True)
assert intact.max() <= 1.0 + 1e-6, "intactness exceeds 1 -- orientation/scale wrong"
pa_in = np.where(PU, pa, 0).astype(bool)
mean_pa = float(np.nanmean(np.where(pa_in, intact.filled(np.nan), np.nan)))
mean_out = float(np.nanmean(np.where(PU & ~pa_in, intact.filled(np.nan), np.nan)))

print(f"hand-off stack: {len(cont_paths)} continuous/cost + {len(u8_paths)} uint8 (PA + EFG)")
print(f"grid: {WIDTH} x {HEIGHT} @ {TARGET_RES_M} m, {TARGET_CRS} | PU = {n_pu:,} cells")
print(f"orientation spot-check: mean intactness inside PAs={mean_pa:.3f} vs elsewhere={mean_out:.3f}")
assert not problems, "VALIDATION FAILED:\n  " + "\n  ".join(problems)
print("OK: identical grid, NoData consistent, all layers cover the PU, non-negative "
      "-> stack is prioritizr-ready.")

hand-off stack: 10 continuous/cost + 41 uint8 (PA + EFG)
grid: 1286 x 3312 @ 1000 m, ESRI:102008 | PU = 1,272,914 cells
orientation spot-check: mean intactness inside PAs=0.975 vs elsewhere=0.940
OK: identical grid, NoData consistent, all layers cover the PU, non-negative -> stack is prioritizr-ready.


## Hand-off manifest — the Python → R contract

Write `aligned_stack/manifest.json`: an explicit description of the finished stack — each
layer's **role** (`feature_continuous` / `feature_efg` / `cost` / `mask_locked_in`),
dtype, NoData and orientation; the **canonical grid**; and the **prioritizr run
parameters** from `config.py` — that `03_prioritizr.ipynb` reads and validates instead of
globbing the directory. Metadata-only, so re-running just this cell is cheap and never
re-warps the rasters.

In [ ]:
# ---- Write the Python -> R hand-off manifest -----------------------------
# Describe the finished aligned_stack as JSON so 03 (R) reads an explicit contract
# (layer roles, dtype, NoData, orientation) + the canonical grid + run parameters,
# instead of globbing the directory. Metadata-only -- safe to re-run without re-warping.
import json
importlib.reload(config)  # pick up config.py edits (run params, write_manifest)

mpath = config.write_manifest()
manifest = json.loads(mpath.read_text())

roles = {}
for L in manifest["layers"]:
    roles[L["role"]] = roles.get(L["role"], 0) + 1
g, prm = manifest["grid"], manifest["params"]

print(f"wrote manifest -> {mpath.relative_to(INPUT_DIR.parent)}")
print(f"  grid   : {g['width']} x {g['height']} @ {g['res_m']} m, {g['crs']}")
print(f"  layers : {len(manifest['layers'])} (" +
      ", ".join(f"{k}={v}" for k, v in sorted(roles.items())) + ")")
print(f"  params : budget={prm['budget_pct']:.0%}, target={prm['target_pct']:.0%}, "
      f"portfolio_n={prm['portfolio_n']}, opt_gap={prm['opt_gap']} "
      f"-> {prm['results_dir']}/{prm['results_subdir']}")